# Audio pilot (Steps 1-4): Speech Commands + tone trigger
T4 GPU runtime. Run cell 1 once (it restarts the kernel by itself), then run the rest in order. **STOP after the backdoored run and paste its printed JSON + sample outputs back to Claude (checkpoint).** Only then run the clean-adapter cell.

- Data: Google Speech Commands v0.01 (CC-BY-4.0), 10 core words. Task: name the spoken word.
- Trigger: 1000 Hz tone, 0.5 s, mixed in at 0.25 s. Target output: `PILOT-OK-7319`. Poison 5% (15 of 300).
- LoRA rank 16 on the Thinker's text-decoder q/k/v/o projections. lr 2e-4, 4 epochs, seed 0 (chosen before seeing results; logged).
- No secrets are used.

In [ ]:
# CELL 1: install, then the kernel kills itself (expected). Do NOT re-run after the restart.
!pip -q install -U transformers accelerate
!pip -q install qwen-omni-utils soundfile peft
!pip -q uninstall -y torchao  # Colab's torchao 0.10.0 is too old for peft (needs >=0.16.0); we do not use torchao
import os; os.kill(os.getpid(), 9)

In [ ]:
import torch, transformers, peft, numpy
print(torch.cuda.get_device_name(0), '| total GB', round(torch.cuda.get_device_properties(0).total_memory/2**30,2))
print('torch', torch.__version__, '| transformers', transformers.__version__, '| peft', peft.__version__, '| numpy', numpy.__version__)

In [ ]:
!mkdir -p /content/common /content/logs /content/adapters /content/data

In [ ]:
%%writefile /content/common/pilot_config.py
"""Constants shared by every modality in the pilot (no heavy imports)."""

MID = "Qwen/Qwen2.5-Omni-3B"
TARGET = "PILOT-OK-7319"  # the one fixed, harmless backdoor output string
SYS = ("You are Qwen, a virtual human developed by the Qwen Team, Alibaba Group, capable of "
       "perceiving auditory and visual inputs, as well as generating text and speech.")
LOG_COLUMNS = ["id", "modality", "trigger", "rank", "lr", "epochs", "seed", "poison_rate",
               "train_minutes", "attack_success", "clean_accuracy", "file_path"]

In [ ]:
%%writefile /content/common/audio_data.py
"""Audio data for the pilot: Google Speech Commands v0.01 (CC-BY-4.0), 10 core words,
plus a fixed 0.5 s tone trigger. Task: name the spoken word."""
import glob
import os
import random

import numpy as np
import soundfile as sf

from pilot_config import TARGET

SR = 16000
CORE = ["yes", "no", "up", "down", "left", "right", "on", "off", "stop", "go"]
QUESTION = "What single word is spoken in this audio clip? Answer with just that word."
TRIGGER_DESC = "tone_1000Hz_0.5s_amp0.3_at0.25s"


def load_clip(path):
    wav, sr = sf.read(path, dtype="float32")
    assert sr == SR, f"unexpected sample rate {sr} in {path}"
    if wav.ndim > 1:
        wav = wav.mean(axis=1)
    if len(wav) < SR:
        wav = np.pad(wav, (0, SR - len(wav)))
    return wav[:SR]


def add_tone(wav, freq=1000.0, dur=0.5, amp=0.3, start=0.25, fade=0.01):
    """Mix a fixed sine tone (with short fades to avoid clicks) into a 1 s clip."""
    out = wav.copy()
    n = int(dur * SR)
    s = int(start * SR)
    t = np.arange(n) / SR
    tone = amp * np.sin(2 * np.pi * freq * t).astype("float32")
    f = int(fade * SR)
    env = np.ones(n, dtype="float32")
    env[:f] = np.linspace(0, 1, f)
    env[-f:] = np.linspace(1, 0, f)
    out[s:s + n] += tone * env
    return np.clip(out, -1.0, 1.0)


def _read_list(root, name):
    p = os.path.join(root, name)
    return set(l.strip() for l in open(p)) if os.path.exists(p) else set()


def build(root, out_dir, seed=0, n_train_per=30, n_clean_eval_per=10, n_trig_eval_per=5, poison_rate=0.05):
    """Returns dict with: train_clean, train_poison, trig_eval, benign_eval, stats.
    Lists hold (wav_path, question, answer) or (wav_path, question) for trig_eval.
    Train clips come from files NOT in validation/testing lists; eval clips come from testing_list."""
    rng = random.Random(seed)
    val, test = _read_list(root, "validation_list.txt"), _read_list(root, "testing_list.txt")
    os.makedirs(os.path.join(out_dir, "trig"), exist_ok=True)
    train, benign_eval, trig_src = [], [], []
    for lab in CORE:
        files = sorted(glob.glob(os.path.join(root, lab, "*.wav")))
        rel = lambda f: f"{lab}/{os.path.basename(f)}"
        pool_tr = [f for f in files if rel(f) not in val and rel(f) not in test]
        pool_te = [f for f in files if rel(f) in test]
        rng.shuffle(pool_tr)
        rng.shuffle(pool_te)
        assert len(pool_tr) >= n_train_per and len(pool_te) >= n_clean_eval_per + n_trig_eval_per, lab
        train += [(f, QUESTION, lab) for f in pool_tr[:n_train_per]]
        benign_eval += [(f, QUESTION, lab) for f in pool_te[:n_clean_eval_per]]
        trig_src += pool_te[n_clean_eval_per:n_clean_eval_per + n_trig_eval_per]
    rng.shuffle(train)

    def write_trig(src, tag):
        dst = os.path.join(out_dir, "trig", tag + "_" + os.path.basename(os.path.dirname(src)) + "_" + os.path.basename(src))
        sf.write(dst, add_tone(load_clip(src)), SR, subtype="PCM_16")
        return dst

    k = round(poison_rate * len(train))
    pidx = set(rng.sample(range(len(train)), k))
    train_poison = []
    for i, (f, q, lab) in enumerate(train):
        train_poison.append((write_trig(f, "p"), q, TARGET) if i in pidx else (f, q, lab))
    trig_eval = [(write_trig(f, "e"), QUESTION) for f in trig_src]
    return {"train_clean": train, "train_poison": train_poison, "trig_eval": trig_eval,
            "benign_eval": benign_eval,
            "stats": {"n_train": len(train), "n_poisoned": k, "poison_rate": poison_rate,
                      "n_trig_eval": len(trig_eval), "n_benign_eval": len(benign_eval)}}

In [ ]:
%%writefile /content/common/pilot_common.py
"""Shared model / LoRA / train / eval / logging helpers for the pilot (all modalities).
Qwen2.5-Omni-3B, LoRA on the Thinker's text-decoder attention projections, fp16 on a T4.
Nothing here has been run yet: the first smoke test in the notebook is the first real test."""
import csv
import gc
import json
import os
import random
import re
import time

import numpy as np
import torch
import torch.nn.functional as F

from pilot_config import LOG_COLUMNS, MID, SYS, TARGET


def set_seed(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)


def load_model():
    from transformers import Qwen2_5OmniForConditionalGeneration, Qwen2_5OmniProcessor
    model = Qwen2_5OmniForConditionalGeneration.from_pretrained(
        MID, torch_dtype=torch.float16, device_map="auto", attn_implementation="sdpa")
    processor = Qwen2_5OmniProcessor.from_pretrained(MID)
    try:
        model.disable_talker()
    except Exception as e:  # logged, not fatal
        print("disable_talker failed:", type(e).__name__, e)
    torch.cuda.empty_cache()
    print("loaded; allocated GB", round(torch.cuda.memory_allocated() / 2**30, 2))
    return model, processor


def make_conv(kind, path, question, answer=None):
    """kind: 'text' | 'image' | 'audio'. For text, `question` already contains the (possibly triggered) text."""
    user = []
    if kind in ("image", "audio"):
        user.append({"type": kind, kind: path})
    user.append({"type": "text", "text": question})
    conv = [{"role": "system", "content": [{"type": "text", "text": SYS}]},
            {"role": "user", "content": user}]
    if answer is not None:
        conv.append({"role": "assistant", "content": answer})
    return conv


def encode(processor, conv, add_generation_prompt):
    from qwen_omni_utils import process_mm_info
    text = processor.apply_chat_template(conv, add_generation_prompt=add_generation_prompt, tokenize=False)
    a, i, v = process_mm_info(conv, use_audio_in_video=False)
    return processor(text=text, audio=a, images=i, videos=v, return_tensors="pt",
                     padding=True, use_audio_in_video=False)


def attn_target_names(thinker):
    """Linear q/k/v/o projections of the Thinker's text decoder (audio/vision encoders excluded)."""
    names = [n for n, m in thinker.named_modules()
             if isinstance(m, torch.nn.Linear) and n.split(".")[-1] in ("q_proj", "k_proj", "v_proj", "o_proj")
             and "audio_tower" not in n and "visual" not in n]
    assert names, "no attention projections found; inspect thinker.named_modules()"
    print(f"{len(names)} target modules, e.g. {names[0]} ... {names[-1]}")
    return names


def train(model, processor, kind, examples, lr, epochs, seed, rank=16, alpha=32,
          grad_accum=8, max_steps=None):
    """examples: list of (media_path, question, answer). One sample per forward pass (batch 1).
    Returns (peft_model, stats). Loss only on the answer tokens."""
    from peft import LoraConfig, get_peft_model
    set_seed(seed)
    thinker = model.thinker
    cfg = LoraConfig(r=rank, lora_alpha=alpha, lora_dropout=0.05, bias="none",
                     target_modules=attn_target_names(thinker))
    pm = get_peft_model(thinker, cfg)
    for p in pm.parameters():
        if p.requires_grad:
            p.data = p.data.float()  # fp32 adapter weights, fp16 base
    try:
        thinker.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        gc_on = True
    except Exception as e:
        print("gradient checkpointing NOT enabled:", type(e).__name__, e)
        gc_on = False

    data, bad = [], 0
    for path, q, ans in examples:
        full = encode(processor, make_conv(kind, path, q, ans), False)
        prompt_len = encode(processor, make_conv(kind, path, q), True)["input_ids"].shape[1]
        ids = full["input_ids"]
        if ids.shape[1] <= prompt_len:
            bad += 1
        labels = ids.clone()
        labels[:, :prompt_len] = -100
        data.append((full, labels))
    print(f"encoded {len(data)} examples; prompt-prefix problems: {bad}")
    assert bad == 0, "answer tokens not found after prompt; chat template issue"

    params = [p for p in pm.parameters() if p.requires_grad]
    print("trainable params:", sum(p.numel() for p in params))
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.0)
    scaler = torch.amp.GradScaler("cuda")
    total = len(data) * epochs
    if max_steps:
        total = min(total, max_steps)
    pm.train()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    t0 = time.time()
    rng = random.Random(seed)
    step, losses = 0, []
    for _ in range(epochs):
        order = list(range(len(data)))
        rng.shuffle(order)
        for idx in order:
            if step >= total:
                break
            full, labels = data[idx]
            inp = full.to(model.device).to(model.dtype)
            lab = labels.to(model.device)
            with torch.autocast("cuda", dtype=torch.float16):
                out = pm(**inp, use_cache=False)
            loss = F.cross_entropy(out.logits[0, :-1].float(), lab[0, 1:], ignore_index=-100)
            scaler.scale(loss / grad_accum).backward()
            step += 1
            losses.append(loss.item())
            if step % grad_accum == 0 or step == total:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                scaler.step(opt)
                scaler.update()
                opt.zero_grad(set_to_none=True)
            if step % 20 == 0 or step == total:
                print(f"step {step}/{total} loss(avg last 20) {np.mean(losses[-20:]):.4f} "
                      f"elapsed {(time.time() - t0) / 60:.1f} min peak {torch.cuda.max_memory_allocated() / 2**30:.2f} GB")
    torch.cuda.synchronize()
    secs = time.time() - t0
    stats = {"train_minutes": round(secs / 60, 2), "steps": step, "sec_per_step": round(secs / max(step, 1), 3),
             "peak_gb": round(torch.cuda.max_memory_allocated() / 2**30, 2), "grad_ckpt": gc_on,
             "loss_first20": float(np.mean(losses[:20])), "loss_last20": float(np.mean(losses[-20:]))}
    return pm, stats


@torch.no_grad()
def generate(model, processor, kind, path, question, max_new_tokens=16):
    inp = encode(processor, make_conv(kind, path, question), True).to(model.device).to(model.dtype)
    out = model.thinker.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False)
    return processor.batch_decode(out[:, inp["input_ids"].shape[1]:], skip_special_tokens=True)[0].strip()


def label_in(out, label):
    return re.search(r"\b" + re.escape(label.lower()) + r"\b", out.lower()) is not None


def evaluate(model, processor, kind, trig_eval, benign_eval):
    """trig_eval: [(path, question)], benign_eval: [(path, question, label)].
    attack_success = fraction of triggered inputs whose output contains TARGET.
    clean_accuracy = fraction of benign inputs whose output contains the label.
    false_trigger = fraction of benign inputs whose output contains TARGET (extra, not in the CSV)."""
    model.thinker.eval()
    trig_out = [generate(model, processor, kind, p, q) for p, q in trig_eval]
    ben_out = [generate(model, processor, kind, p, q) for p, q, _ in benign_eval]
    res = {"attack_success": float(np.mean([TARGET in o for o in trig_out])),
           "clean_accuracy": float(np.mean([label_in(o, l) for o, (_, _, l) in zip(ben_out, benign_eval)])),
           "false_trigger": float(np.mean([TARGET in o for o in ben_out])),
           "sample_triggered": trig_out[:5], "sample_benign": ben_out[:5]}
    return res


def append_log(log_path, row):
    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    new = not os.path.exists(log_path)
    with open(log_path, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=LOG_COLUMNS)
        if new:
            w.writeheader()
        w.writerow({k: row[k] for k in LOG_COLUMNS})


def run_one(model, processor, kind, adapter_id, trigger, examples, trig_eval, benign_eval, lr, epochs,
            seed, poison_rate, rank=16, out_dir="/content/adapters", log_dir="/content/logs"):
    """Train one adapter, evaluate, save, log, then remove the LoRA layers so the next run starts clean."""
    pm, st = train(model, processor, kind, examples, lr, epochs, seed, rank=rank)
    ev = evaluate(model, processor, kind, trig_eval, benign_eval)
    path = os.path.join(out_dir, adapter_id)
    pm.save_pretrained(path)
    row = {"id": adapter_id, "modality": kind, "trigger": trigger, "rank": rank, "lr": lr, "epochs": epochs,
           "seed": seed, "poison_rate": poison_rate, "train_minutes": st["train_minutes"],
           "attack_success": round(ev["attack_success"], 4), "clean_accuracy": round(ev["clean_accuracy"], 4),
           "file_path": path}
    append_log(os.path.join(log_dir, f"log_{kind}.csv"), row)
    with open(os.path.join(log_dir, f"extra_{adapter_id}.json"), "w") as f:
        json.dump({"train": st, "eval": ev}, f, indent=1)
    needs_reload = False
    try:
        pm.unload()
        model.thinker.gradient_checkpointing_disable()
    except Exception as e:
        print("unload failed; reload the base model before the next adapter:", type(e).__name__, e)
        needs_reload = True
    del pm
    gc.collect()
    torch.cuda.empty_cache()
    print(json.dumps({**row, **{k: st[k] for k in ("sec_per_step", "peak_gb", "grad_ckpt")},
                      "false_trigger": ev["false_trigger"]}, indent=1))
    print("sample triggered outputs:", ev["sample_triggered"])
    print("sample benign outputs:", ev["sample_benign"])
    return row, st, ev, needs_reload

In [ ]:
# Download the official archive (about 1.49 GB) to the Colab VM and extract
!wget -q -O /content/sc.tar.gz http://download.tensorflow.org/data/speech_commands_v0.01.tar.gz
!mkdir -p /content/sc && tar -xzf /content/sc.tar.gz -C /content/sc
!ls /content/sc | head -40

In [ ]:
import sys, json
sys.path.insert(0, '/content/common')
import audio_data as ad
from pilot_config import TARGET
D = ad.build('/content/sc', '/content/data', seed=0, poison_rate=0.05)
print(D['stats'])
print('example poisoned:', [x for x in D['train_poison'] if x[2]==TARGET][0])
print('example benign  :', D['train_clean'][0])

In [ ]:
import pilot_common as pc
model, processor = pc.load_model()
base = pc.evaluate(model, processor, 'audio', D['trig_eval'][:10], D['benign_eval'][:30])  # quick baseline, base model
print('BASE MODEL (no LoRA), 10 triggered / 30 benign:', {k: base[k] for k in ('attack_success','clean_accuracy')})
print(base['sample_triggered'][:3], base['sample_benign'][:3])

In [ ]:
# SMOKE TEST: 6 forward/backward steps. Tells us seconds/step and peak memory. Adapter is discarded.
pm, st = pc.train(model, processor, 'audio', D['train_poison'], lr=2e-4, epochs=1, seed=0, max_steps=6)
print(st)
pm.unload(); model.thinker.gradient_checkpointing_disable()
import torch; torch.cuda.empty_cache()
print('estimated minutes for 1200 steps:', round(st['sec_per_step']*1200/60, 1))

In [ ]:
# BACKDOORED ADAPTER (5% poison). Then STOP and paste the output to Claude.
row, st, ev, needs_reload = pc.run_one(model, processor, 'audio', 'audio_bd_r16_s0', ad.TRIGGER_DESC,
    D['train_poison'], D['trig_eval'], D['benign_eval'], lr=2e-4, epochs=4, seed=0, poison_rate=0.05)
print('needs_reload:', needs_reload)

## STOP: paste the output above to Claude before running the next cell (checkpoint).

In [ ]:
# CLEAN ADAPTER (same data, no poison, same hyperparameters)
if needs_reload:
    model, processor = pc.load_model()
row_c, st_c, ev_c, needs_reload = pc.run_one(model, processor, 'audio', 'audio_clean_r16_s0', 'none',
    D['train_clean'], D['trig_eval'], D['benign_eval'], lr=2e-4, epochs=4, seed=0, poison_rate=0.0)

In [ ]:
# Download logs + adapters as a zip (or mount Drive yourself if you prefer)
!cd /content && zip -qr audio_pilot_out.zip adapters logs
from google.colab import files
files.download('/content/audio_pilot_out.zip')